# PSX Due-Diligence Agent — Qwen (DashScope)

Router graph: **init_node (router)** classifies the query, then fans out to **compliance** and/or **finance** (in parallel), and **synthesize** merges multi-agent answers into one reply.

Run cells top to bottom. To start a **fresh conversation**, re-run the *thread id* then *context* cells. To ask a new question, edit the *user input* cell and run *invoke*. Ensure `.env` has `DASHSCOPE_API_KEY`; select the `.venv` kernel.

In [ ]:
import os
import uuid
from dotenv import load_dotenv

load_dotenv(override=True)  # reads .env; override=True so .env wins over any stale system env vars
os.environ.setdefault("DB_PATH", os.path.join(os.getcwd(), "data", "psx.db"))
# Finance agent reads markdown summaries from FINANCE_DATA_DIR (set it in .env once you
# move the data in; for now default to the existing finance project folder).
os.environ.setdefault("FINANCE_DATA_DIR", r"C:\Users\Pari\Documents\finance\data_md")

from app.common.utils import _get_model
from app.common.context import SessionContext
from app.graph.compile import get_compiled_agent

assert os.getenv("DASHSCOPE_API_KEY"), "Set DASHSCOPE_API_KEY in .env (see .env.example)"
print("DB_PATH          =", os.environ["DB_PATH"])
print("FINANCE_DATA_DIR =", os.environ["FINANCE_DATA_DIR"])
print("model            =", os.getenv("QWEN_MODEL", "qwen-plus"))

DB_PATH          = data/psx.db
FINANCE_DATA_DIR = C:\Users\Pari\Documents\finance\data_md
model            = qwen-flash


In [ ]:
# --- LangSmith tracing (EU region) ---
# Your key is an EU-region key, so use the EU endpoint. Add LANGSMITH_API_KEY to .env.
# Use the canonical LANGSMITH_* names only (do NOT also set LANGCHAIN_ENDPOINT -- mixing
# the two causes endpoint conflicts). Run this BEFORE the first invoke in a fresh kernel.
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGSMITH_PROJECT"] = "devpost-hackathon"

assert os.getenv("LANGSMITH_API_KEY"), "Add LANGSMITH_API_KEY (EU region) to .env"
print("LangSmith tracing ON -> project:", os.environ["LANGSMITH_PROJECT"], "| endpoint: EU")

LangSmith tracing ON -> project: devpost-hackathon | endpoint: EU


In [3]:
# Compile the full PSX graph once (router -> compliance/finance -> synthesize).
agent = await get_compiled_agent("psx_agent")

In [ ]:
# Thread id -- re-run this cell to start a brand-new conversation (fresh memory).
thread_id = str(uuid.uuid4())
print("thread_id =", thread_id) 

thread_id = a3bd3051-dbd0-4845-86aa-ea0f36f92bc2


In [5]:
# Context -- edit freely (e.g. swap model tier). Re-run after changing thread_id above.
context = SessionContext(
    thread_id=thread_id,
    model=_get_model("qwen"),
    agents=None,  
    
)
config = {"configurable": {"thread_id": thread_id}}

In [11]:
# User input -- edit this, then run the invoke cell below.
user_input = "which company had higest profit in last year"

In [13]:
# Invoke the graph with the user input + context, streaming the answer.
# The graph emits chunks per source: compliance_chunk / finance_chunk (specialists working)
# and synthesize_chunk (the merged answer, only for multi-agent queries).
async for chunk in agent.astream(
    {"messages": [{"role": "user", "content": user_input}]},
    stream_mode="custom",
    context=context,
    config=config,
):  
    # print(context.agents)
    if context.agents == ["compliance_node"]:
        content = chunk.get('compliance_chunk')
    elif context.agents == ["finance_node"]:
        content = chunk.get('finance_chunk')
    else:  # multi-agent or synthesize node emitting
        content = chunk.get('synthesize_chunk')
    if content:
        print(content, end="", flush=True)
print()

I can help you identify which company had the highest profit in the last fiscal year based on available financial data, but I cannot give investment advice or recommend specific stocks.

Let me check which companies and years have financial data available, and then we can look at the most recent profit figures.

I'm sorry, but there is no financial data available in the system at this time. This means I cannot provide information about which company had the highest profit in the last fiscal year or any other financial details.

Please ensure that the financial summary files for companies (in markdown format) are properly placed in the `C:\Users\Pari\Documents\finance\data_md` directory, and that they include data for recent fiscal years (e.g., 2021–2025).

Once the data is available, I can help you analyze it — for example, by showing which company reported the highest net profit, comparing performance across years, or explaining key financial metrics like revenue, operating profit, or